In [2]:
import sys, time
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import functions as F

from src.config import load_config
from src.spark_session import get_spark
from src.evaluation.metrics import evaluate
from src.evaluation.stats import paired_bootstrap
from src.models.popularity import add_recent_counts, smoothed_ctr
from src.models.embeddings import user_profiles_dense, score_dense
from src.models.hybrid import zscore, weighted
from src.models.als import build_interactions, fit_indexers, add_indices, fit_als, score_als

cfg = load_config("small")
cfg["spark"]["master"] = "local[6]"     # menos procesos de Python simultáneos
spark = get_spark(cfg)
spark.sparkContext.setCheckpointDir(str(ROOT / "data" / "checkpoints"))
OUT = cfg["paths"]["processed"]
RES = ROOT / "results" / "metrics"
FIG = cfg["paths"]["figures"]

# Configuraciones fijadas en val
W_HIBRIDO = 0.4          # puntaje z
ALS_RANK, ALS_ALPHA = 32, 10
DELTA = 0.01
T_INICIO = time.time()

resultados, por_imp = [], {}

def evaluar(df, expr, nombre):
    """Evalúa, guarda el resumen y las métricas por impresión (para bootstrap y segmentos)."""
    t0 = time.time()
    per_imp, res = evaluate(df.withColumn("score", expr))
    por_imp[nombre] = per_imp.select("impression_id", "n", "auc", "mrr", "ndcg5", "ndcg10").toPandas()
    per_imp.unpersist()
    resultados.append({"modelo": nombre, **res})
    print(f"{nombre:14s} AUC = {res['auc']:.4f} | MRR = {res['mrr']:.4f} | "
          f"nDCG@5 = {res['ndcg5']:.4f} ({time.time() - t0:.0f} s)")

In [3]:
t0 = time.time()
imp = spark.read.parquet(str(OUT / "impressions_long"))
beh = spark.read.parquet(str(OUT / "behaviors"))

p0 = imp.filter(F.col("periodo").isin("train", "val")).agg(F.avg("label")).first()[0]

# Test (15 nov) + las 24 h previas (14 nov = periodo val)
base = (imp.filter(F.col("periodo").isin("val", "test"))
           .select("impression_id", "user_id", "time", "news_id", "label", "periodo"))
cands = (add_recent_counts(base, hours=(24,))
         .filter("periodo = 'test'")
         .withColumn("ctr", smoothed_ctr(24, 20, p0))
         .select("impression_id", "user_id", "news_id", "label", "ctr"))

users_test = (beh.filter("periodo = 'test'")
                 .select("user_id", "history", "hist_len")
                 .dropDuplicates(["user_id"]).cache())

print(f"p0 (9–14 nov) = {p0:.4f} ({time.time() - t0:.0f} s)")

p0 (9–14 nov) = 0.0404 (7 s)


In [ ]:
t0 = time.time()
news_emb = spark.read.parquet(str(OUT / "news_emb_minilm")).cache()
profiles = user_profiles_dense(users_test, news_emb, k=None)

senales = (score_dense(cands, news_emb, profiles)
           .withColumn("emb_resp", F.coalesce("cos", "ctr"))        # embeddings, CTR si no hay historial
           .withColumn("ctr_z", zscore("ctr"))
           .withColumn("cos_z", zscore("cos"))
           .withColumn("hibrido", weighted("cos_z", "ctr_z", W_HIBRIDO))
           .select("impression_id", "user_id", "news_id", "label", "ctr", "emb_resp", "hibrido")
           .cache())

print(f"Candidatas de test: {senales.count():,} ({time.time() - t0:.0f} s)")

In [ ]:
t0 = time.time()
inter = build_interactions(beh, imp, hist_periods=("train", "val", "test"),
                           click_periods=("train", "val")).cache()
u_idx, i_idx = fit_indexers(inter)
n_i = len(i_idx.labelsArray[0])
inter_idx = add_indices(inter, u_idx, i_idx).select("u", "i", "r").cache()
model = fit_als(inter_idx, rank=ALS_RANK, alpha=ALS_ALPHA)

cands_idx = add_indices(cands.select("impression_id", "user_id", "news_id", "label"), u_idx, i_idx)
als_scored = (score_als(model, cands_idx, cold="neutral")
              .withColumn("item_frio", F.col("i") == n_i)
              .select("impression_id", "label", "score_als", "item_frio")
              .cache())

als_scored.agg(F.round(F.avg(F.col("item_frio").cast("int")), 3).alias("pct_candidatas_frias"),
               F.round(F.sum(F.col("label") * F.col("item_frio").cast("int")) / F.sum("label"), 3)
                .alias("pct_clics_frios")).show()
print(f"Celda 4: {time.time() - t0:.0f} s")

In [ ]:
evaluar(senales, F.rand(1), "aleatorio")
evaluar(senales, F.col("ctr"), "baseline_ctr")
evaluar(senales, F.col("emb_resp"), "embeddings")
evaluar(als_scored, F.col("score_als"), "als")
evaluar(senales, F.col("hibrido"), "hibrido")

tabla = (pd.DataFrame(resultados)[["modelo", "auc", "mrr", "ndcg5", "ndcg10", "impresiones"]]
         .sort_values("auc", ascending=False).reset_index(drop=True))
print(f"\nTiempo acumulado: {(time.time() - T_INICIO) / 60:.1f} min")
tabla.round(4)

In [ ]:
m = por_imp["hibrido"].merge(por_imp["baseline_ctr"], on="impression_id", suffixes=("_h", "_b"))

filas_bs = []
for met in ("auc", "mrr", "ndcg5", "ndcg10"):
    dif, ic, medias = paired_bootstrap(m[f"{met}_h"], m[f"{met}_b"])
    filas_bs.append({"metrica": met, "diferencia": dif, "ic_inf": ic[0], "ic_sup": ic[1],
                     "p_dif_menor_igual_0": np.mean(medias <= 0),
                     "p_dif_menor_igual_delta": np.mean(medias <= DELTA)})
bootstrap = pd.DataFrame(filas_bs)

auc_bs = bootstrap.iloc[0]
print(f"H1 (AUC híbrido − AUC baseline ≥ {DELTA}): "
      f"{'SE ACEPTA' if auc_bs['ic_inf'] > DELTA else 'NO SE ACEPTA'} "
      f"(IC 95 % = [{auc_bs['ic_inf']:+.4f}, {auc_bs['ic_sup']:+.4f}])")
bootstrap.round(4)

In [ ]:
# Usuarios presentes en entrenamiento (como Overlap/Unseen Users del paper de MIND)
usr_train = (beh.filter(F.col("periodo").isin("train", "val"))
                .select("user_id").distinct().withColumn("en_train", F.lit(True)))

# Impresiones donde todos los clics caen en noticias desconocidas para ALS
clic_frio = (als_scored.filter("label = 1").groupBy("impression_id")
                       .agg(F.min(F.col("item_frio").cast("int")).alias("todos_clics_frios")))

seg = (beh.filter("periodo = 'test'")
          .select("impression_id", "user_id", "hist_len", "imp_len")
          .join(F.broadcast(usr_train), "user_id", "left")
          .join(clic_frio, "impression_id", "left")
          .toPandas())
seg["usuario"] = np.where(seg["en_train"].fillna(False).astype(bool), "en entrenamiento", "nuevo")
seg["historial"] = pd.cut(seg["hist_len"], [-1, 0, 10, 30, 10**6],
                          labels=["0", "1–10", "11–30", ">30"]).astype(str)
seg["clics"] = np.where(seg["todos_clics_frios"] == 1, "todos en noticias nuevas", "alguno conocido")

modelos = ["baseline_ctr", "embeddings", "als", "hibrido"]
largo = (pd.concat([por_imp[mo][["impression_id", "auc"]].assign(modelo=mo) for mo in modelos])
           .merge(seg, on="impression_id"))

tablas_seg = {}
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for ax, dim in zip(axes, ["usuario", "historial", "clics"]):
    t = largo.pivot_table(index=dim, columns="modelo", values="auc", aggfunc="mean")[modelos]
    t["impresiones"] = seg.groupby(dim).size()
    tablas_seg[dim] = t
    t[modelos].plot.bar(ax=ax, rot=0)
    ax.set_ylim(0.45, 0.75); ax.axhline(0.5, color="k", lw=0.6, ls="--")
    ax.set_title(f"AUC por {dim}"); ax.set_xlabel(""); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIG / "07_auc_por_segmento.png", dpi=150); plt.show()

for dim, t in tablas_seg.items():
    print(f"\n== {dim} ==")
    print(t.round(4).to_string())

In [4]:
INCLUDE_TFIDF = True      # cambia a True si tienes 15–25 minutos más

if INCLUDE_TFIDF:
    from src.models.content import fit_tfidf, user_profiles, score_content
    t0 = time.time()
    news = spark.read.parquet(str(OUT / "news"))
    model_t, vectorizer = fit_tfidf(news, min_df=5)
    news_vec = model_t.transform(news).select("news_id", "tokens", "vec").cache()
    prof_t = user_profiles(users_test, news_vec, vectorizer, k=None)
    tf = score_content(cands, news_vec, prof_t).withColumn("tfidf_resp", F.coalesce("cos", "ctr"))
    evaluar(tf, F.col("tfidf_resp"), "tfidf")
    tabla = (pd.DataFrame(resultados)[["modelo", "auc", "mrr", "ndcg5", "ndcg10", "impresiones"]]
             .sort_values("auc", ascending=False).reset_index(drop=True))
    print(f"TF-IDF: {time.time() - t0:.0f} s")

Py4JJavaError: An error occurred while calling o817.collectToPython.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 5 in stage 23.0 failed 1 times, most recent failure: Lost task 5.0 in stage 23.0 (TID 133) (KAREN-ASUS executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed). Consider setting 'spark.sql.execution.pyspark.udf.faulthandler.enabled' or'spark.python.worker.faulthandler.enabled' configuration to 'true' for the better Python traceback.
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:621)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:599)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:35)
	at org.apache.spark.sql.execution.python.BasePythonUDFRunner$$anon$2.read(PythonUDFRunner.scala:112)
	at org.apache.spark.sql.execution.python.BasePythonUDFRunner$$anon$2.rea

In [ ]:
# SI CORRI SOLO TF-IDF
prev = pd.read_csv(RES / "07_test_final.csv")
fila = (pd.DataFrame([r for r in resultados if r["modelo"] == "tfidf"])
          .assign(periodo="test", dataset=cfg["dataset"]))[prev.columns]
(pd.concat([prev[prev["modelo"] != "tfidf"], fila])
   .sort_values("auc", ascending=False)
   .to_csv(RES / "07_test_final.csv", index=False))
print(pd.read_csv(RES / "07_test_final.csv").round(4))
spark.stop()

In [ ]:
tabla.assign(periodo="test", dataset=cfg["dataset"]).to_csv(RES / "07_test_final.csv", index=False)
bootstrap.to_csv(RES / "07_bootstrap_hibrido_vs_baseline.csv", index=False)
for dim, t in tablas_seg.items():
    t.to_csv(RES / f"07_segmentos_{dim}.csv")
pd.concat([d.assign(modelo=k) for k, d in por_imp.items()]) \
  .to_parquet(RES / "07_metricas_por_impresion.parquet", index=False)

print(f"Tiempo total: {(time.time() - T_INICIO) / 60:.1f} min")
spark.stop()